In [1]:
import torch
import sklearn
import numpy as np
import pandas as pd
from pytorch_pretrained_bert import BertTokenizer, BertModel, BertAdam, BertForSequenceClassification
from pytorch_pretrained_bert.file_utils import PYTORCH_PRETRAINED_BERT_CACHE

import os
import csv
import logging
from torch.utils.data import TensorDataset, DataLoader, RandomSampler, SequentialSampler
from torch.utils.data.distributed import DistributedSampler
from tqdm import tqdm, trange

In [2]:
local_rank = -1
bert_model_name = 'bert-base-uncased'

In [3]:
# define tokenizer
bert_tok = BertTokenizer.from_pretrained(bert_model_name)

If GPU devices are available, we should specify that <br>
- we want to use GPU <br>
- how many GPU we do have <br>

to `torch`

In [4]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
n_gpu = torch.cuda.device_count()

`text` will be tokenized as `tokenized_text` and will be converted to `indexed_tokens` by `bert_tok` tokenizer object

In [5]:
# Tokenized input
text = "[CLS] Who was Jim Henson ? [SEP] Jim Henson was a puppeteer [SEP]"
tokenized_text = bert_tok.tokenize(text)
tokenized_text

['[CLS]',
 'who',
 'was',
 'jim',
 'henson',
 '?',
 '[SEP]',
 'jim',
 'henson',
 'was',
 'a',
 'puppet',
 '##eer',
 '[SEP]']

In [6]:
# Convert token to vocabulary indices
indexed_tokens = bert_tok.convert_tokens_to_ids(tokenized_text)

# Define sentence A and B indices associated to 1st and 2nd sentences (see paper)
segments_ids = [0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 1, 1, 1, 1]
indexed_tokens

[101,
 2040,
 2001,
 3958,
 27227,
 1029,
 102,
 3958,
 27227,
 2001,
 1037,
 13997,
 11510,
 102]

Load model

In [7]:
aux_columns = [
    'target',
    'severe_toxicity',
    'obscene',
    'identity_attack',
    'insult',
    'threat',
    'sexual_explicit',
]

identity_columns = [
    'male',
    'female',
    'homosexual_gay_or_lesbian',
    'christian',
    'jewish',
    'muslim',
    'black',
    'white',
    'psychiatric_or_mental_illness'
]

coll = [
    'black',
    'white',
    'homosexual_gay_or_lesbian',
    'muslim'
]

In [8]:
num_labels = len(aux_columns)
model = BertForSequenceClassification.from_pretrained(
    bert_model_name,
    cache_dir=PYTORCH_PRETRAINED_BERT_CACHE / 'distributed_{}'.format(local_rank),
    num_labels = num_labels)

Prepare dataloader

To build dataloader, we need to define following variables

In [9]:
DATA_DIR = '/data/jigsaw/glue_data/JIG'
MODEL_DIR = '/data/jigsaw/model'
MAX_LEN = 80
BATCH_SIZE = 32
EPOCHS = 5
NUM_TRAIN_EPOCHS = 2

To build dataloader, we need several classes to represent our dataset.

`InputExample` class is an instance for containing a real text and label for it.<br>
`InputFeatures` class is an instance for containing preprocessed inputs and label for the corresponding inputs.<br>
`DataProcessor` class is parent class of `JigsawProcessor`.<br>
`JigsawProcessor` class is an actual class that represents the jigsaw dataset.

In [10]:
class InputExample(object):
    """A single training/test example for simple sequence classification."""

    def __init__(self, guid, text, label=None):
        """Constructs a InputExample.
        Args:
            guid: Unique id for the example.
            text_a: string. The untokenized text of the first sequence. For single
            sequence tasks, only this sequence must be specified.
            text_b: (Optional) string. The untokenized text of the second sequence.
            Only must be specified for sequence pair tasks.
            label: (Optional) string. The label of the example. This should be
            specified for train and dev examples, but not for test examples.
        """
        self.guid = guid
        self.text = text
        self.label = label
        
    def __repr__(self):
        return '{} → {} for {}'.format(self.guid, self.label, self.text[:10])

In [11]:
class InputFeatures(object):
    """A single set of features of data."""

    def __init__(self, input_ids, input_mask, segment_ids, label_id):
        self.input_ids = input_ids
        self.input_mask = input_mask
        self.segment_ids = segment_ids
        self.label_id = label_id

In [12]:
class DataProcessor(object):
    """Base class for data converters for sequence classification data sets."""

    def get_train_examples(self, data_dir):
        """Gets a collection of `InputExample`s for the train set."""
        raise NotImplementedError()

    def get_dev_examples(self, data_dir):
        """Gets a collection of `InputExample`s for the dev set."""
        raise NotImplementedError()

    def get_dev_examples_from_csv(self, filename):
        """Gets a collection of `InputExample`s for the dev set."""
        raise NotImplementedError()
        
    def get_labels(self):
        """Gets the list of labels for this data set."""
        raise NotImplementedError()

    @classmethod
    def _read_tsv(cls, input_file, quotechar=None):
        """Reads a tab separated value file."""
        with open(input_file, "r", encoding='utf-8') as f:
            reader = csv.reader(f, delimiter="\t", quotechar=quotechar)
            lines = []
            for line in reader:
                lines.append(line)
            return lines

In [13]:
class JigsawProcessor(DataProcessor):
    """Processor for the RTE data set (GLUE version)."""
    def get_train_examples(self, data_dir):
        """See base class."""
        return self._create_examples(
            self._read_tsv(os.path.join(data_dir, "train.tsv")), "train")

    def get_dev_examples(self, data_dir):
        """See base class."""
        return self._create_examples(
            self._read_tsv(os.path.join(data_dir, "dev.tsv")), "dev")

    def get_examples_from_df(self, df):
        """See base class."""
        for f in identity_columns:
            df.loc[:, f] = np.where(df[f]>=0.5, True, False).astype(int)
            
        lines = [['dummy']]
        for i in trange(df.shape[0]):
            lines.append(list(map(str, df.iloc[i][['target', 'target']+identity_columns+['comment_text']].values)))
        return self._create_examples(lines, 'dev')
    
    def get_dev_examples_from_csv(self, filename):
        """See base class."""
        try:
            df = pd.read_csv(filename)
        except:
            df = pd.read_csv(filename, compression='zip')
        for f in identity_columns:
            df.loc[:, f] = np.where(df[f]>=0.5, True, False).astype(int)
            
        lines = [['dummy']]
        for i in range(df.shape[0]):
            lines.append(list(map(str, df.iloc[i][['target', 'target']+identity_columns+['comment_text']].values)))
        return self._create_examples(lines, 'dev')
    
    def get_labels(self):
        """See base class."""
        return ['0', '1']

    def _create_examples(self, lines, set_type):
        """Creates examples for the training and dev sets."""
        examples = []
        for (i, line) in enumerate(lines):
            if i == 0:
                continue
            try:
                guid = "%s-%s" % (set_type, i)
                text = line[-1]
                label = str(0 if float(line[1]) < 0.5 else 1)
                examples.append(
                    InputExample(guid=guid, text=text, label=label))
            except Exception as e:
                continue
        return examples


`convert_examples_to_features` is a function that actually performs preprocessing for jigsaw dataset

In [14]:
def convert_examples_to_features(examples, label_list, max_seq_length, tokenizer):
    """Loads a data file into a list of `InputBatch`s."""

    label_map = {label : i for i, label in enumerate(label_list)}

    features = []
    for (ex_index, example) in enumerate(tqdm(examples)):
        tokens = tokenizer.tokenize(example.text)
        
        # Account for [CLS] and [SEP] with "- 2"
        if len(tokens) > max_seq_length - 2:
            tokens = tokens[:(max_seq_length - 2)]

        # The convention in BERT is:
        # (a) For sequence pairs:
        #  tokens:   [CLS] is this jack ##son ##ville ? [SEP] no it is not . [SEP]
        #  type_ids: 0   0  0    0    0     0       0 0    1  1  1  1   1 1
        # (b) For single sequences:
        #  tokens:   [CLS] the dog is hairy . [SEP]
        #  type_ids: 0   0   0   0  0     0 0
        #
        # Where "type_ids" are used to indicate whether this is the first
        # sequence or the second sequence. The embedding vectors for `type=0` and
        # `type=1` were learned during pre-training and are added to the wordpiece
        # embedding vector (and position vector). This is not *strictly* necessary
        # since the [SEP] token unambigiously separates the sequences, but it makes
        # it easier for the model to learn the concept of sequences.
        #
        # For classification tasks, the first vector (corresponding to [CLS]) is
        # used as as the "sentence vector". Note that this only makes sense because
        # the entire model is fine-tuned.
        tokens = ["[CLS]"] + tokens + ["[SEP]"]
        segment_ids = [0] * len(tokens)
        input_ids = tokenizer.convert_tokens_to_ids(tokens)

        # The mask has 1 for real tokens and 0 for padding tokens. Only real
        # tokens are attended to.
        input_mask = [1] * len(input_ids)

        # Zero-pad up to the sequence length.
        padding = [0] * (max_seq_length - len(input_ids))
        input_ids += padding
        input_mask += padding
        segment_ids += padding

        assert len(input_ids) == max_seq_length
        assert len(input_mask) == max_seq_length
        assert len(segment_ids) == max_seq_length

        # print several examples
        label_id = label_map[example.label]
        if ex_index < 5:
            print("*** Example ***")
            print("guid: {}".format(example.guid))
            print("tokens: {}".format(" ".join([str(x) for x in tokens])))
            print("input_ids: {}".format(" ".join([str(x) for x in input_ids])))
            print("input_mask: {}".format(" ".join([str(x) for x in input_mask])))
            print("segment_ids: {}".format(" ".join([str(x) for x in segment_ids])))
            print("label: {} (id = {})".format(example.label, label_id))
            print('--------------------------------')

        features.append(
                InputFeatures(input_ids=input_ids,
                              input_mask=input_mask,
                              segment_ids=segment_ids,
                              label_id=label_id))
    return features

When you want to build another BERT model, add `xxxProcessor` in the `processors`

In [15]:
processors = {
    'jigsaw': JigsawProcessor
}
processor = processors['jigsaw']()

In [16]:
train_filename = '/data/jigsaw/train.csv.zip'
#train_filename = '/data/jigsaw/train_sample.csv'
try:
    df = pd.read_csv(train_filename)
except:
    df = pd.read_csv(train_filename, compression='zip')
df = df.sample(frac=1.0)

In [17]:
n_train = int(df.shape[0] * 0.8)
train_df, valid_df = df[:n_train], df[n_train:]
train_df.shape, valid_df.shape

((1443899, 45), (360975, 45))

In [18]:
train_examples = processor.get_examples_from_df(train_df)
label_list = processor.get_labels()

/usr/local/lib/python3.5/dist-packages/pandas/core/indexing.py:543: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: http://pandas.pydata.org/pandas-docs/stable/indexing.html#indexing-view-versus-copy
  self.obj[item] = s


In [19]:
train_features = convert_examples_to_features(train_examples, label_list, MAX_LEN, bert_tok)

*** Example ***
guid: dev-1
tokens: [CLS] ro ##go ##ff owes g ##ci 1 . 4 million dollars and you say fc ##i is a bully . come on and ##i . this is business . g ##ci doesn ' t owe ro ##go ##ff anything . don ' t you think g ##ci is warrant ##ed in collecting a leg ##it debt of such magnitude ? [SEP]
input_ids: 101 20996 3995 4246 24381 1043 6895 1015 1012 1018 2454 6363 1998 2017 2360 4429 2072 2003 1037 20716 1012 2272 2006 1998 2072 1012 2023 2003 2449 1012 1043 6895 2987 1005 1056 12533 20996 3995 4246 2505 1012 2123 1005 1056 2017 2228 1043 6895 2003 10943 2098 1999 9334 1037 4190 4183 7016 1997 2107 10194 1029 102 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0
input_mask: 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0
segment_ids: 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0

Setting some necessary parameters before training our model

FIXME
`weights` will be used for ......

In [20]:
weights = np.ones((len(train_examples),)) / 4
# Subgroup  identity_columns  > 0.5
weights += (train_df[identity_columns].fillna(0).values>=0.5).sum(axis=1).astype(bool).astype(np.int) / 4
# Background Positive, Subgroup Negative
weights += (( (train_df['target'].values>=0.5).astype(bool).astype(np.int) +
   (train_df[identity_columns].fillna(0).values<0.5).sum(axis=1).astype(bool).astype(np.int) ) > 1 ).astype(bool).astype(np.int) / 4
# Background Negative, Subgroup Positive
weights += (( (train_df['target'].values<0.5).astype(bool).astype(np.int) +
   (train_df[identity_columns].fillna(0).values>=0.5).sum(axis=1).astype(bool).astype(np.int) ) > 1 ).astype(bool).astype(np.int) / 4

weights += (( (train_df['target'].values>=0.5).astype(bool).astype(np.int) +
   (train_df[coll].fillna(0).values<0.5).sum(axis=1).astype(bool).astype(np.int) ) > 1 ).astype(bool).astype(np.int) /8

weights += (( (train_df['target'].values<0.5).astype(bool).astype(np.int) +
   (train_df[coll].fillna(0).values>=0.5).sum(axis=1).astype(bool).astype(np.int) ) > 1 ).astype(bool).astype(np.int) / 8

loss_weight = 1.0 / weights.mean()
weights = weights.reshape(-1,1)

In [21]:
NUM_TRAIN_STEP = int(len(train_examples) / BATCH_SIZE * NUM_TRAIN_EPOCHS)
NUM_TRAIN_STEP

90243

`all_input_ids`, `all_input_mask`, `all_segment_ids` will be used for input data of our BERT model <br>
`all_label_ids` is corresonding labels for it

In [22]:
all_input_ids = torch.tensor([f.input_ids for f in train_features], dtype=torch.long)
all_input_mask = torch.tensor([f.input_mask for f in train_features], dtype=torch.long)
all_segment_ids = torch.tensor([f.segment_ids for f in train_features], dtype=torch.long)
all_label_ids = torch.tensor([f.label_id for f in train_features], dtype=torch.long)

In [23]:
print(all_input_ids.shape)
print(all_input_ids.shape)
print(all_input_ids.shape)

torch.Size([1443899, 80])
torch.Size([1443899, 80])
torch.Size([1443899, 80])


In [24]:
train_data = TensorDataset(all_input_ids, all_input_mask, all_segment_ids, all_label_ids)
train_sampler = RandomSampler(train_data)

Finally, we can build our `train_dataloader`

In [25]:
train_dataloader = DataLoader(train_data, sampler=train_sampler, batch_size=BATCH_SIZE)

<b> Train model <b>

set warmup scheduler and other training parameters

In [26]:
def warmup_linear(x, warmup=0.002):
    if x < warmup:
        return x/warmup
    return 1.0 - x

In [27]:
learning_rate = 2e-5
warmup_proportion = 0.1
gradient_accumulation_steps = 1
fp16 = False
global_step = 0
nb_tr_steps = 0
tr_loss = 0
t_total = NUM_TRAIN_STEP

Convert our `model` to GPU-available model

In [28]:
model.train()
if fp16:
    model.half()
model.to(device)

BertForSequenceClassification(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): BertLayerNorm()
      (dropout): Dropout(p=0.1)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0): BertLayer(
          (attention): BertAttention(
            (self): BertSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): BertLayerNorm()
              (dropout): Dropout(p=0.1)
            )
          )
          (intermediat

In [29]:
# Prepare optimizer
param_optimizer = list(model.named_parameters())
no_decay = ['bias', 'LayerNorm.bias', 'LayerNorm.weight']
optimizer_grouped_parameters = [
    {'params': [p for n, p in param_optimizer if not any(nd in n for nd in no_decay)], 'weight_decay': 0.01},
    {'params': [p for n, p in param_optimizer if any(nd in n for nd in no_decay)], 'weight_decay': 0.0}
    ] 

In [30]:
# Create Optimizer
optimizer = BertAdam(optimizer_grouped_parameters,
                     lr=learning_rate,
                     warmup=warmup_proportion,
                     t_total=t_total)

In [36]:
t = trange(NUM_TRAIN_EPOCHS, desc="Epoch")
for epoch in t:
    # If you really want to train your model, comment out the following `break` statement
    # OR, we will just exit this `for` statement
    # break
    
    tr_loss = 0
    nb_tr_examples, nb_tr_steps = 0, 0
    for step, batch in enumerate(tqdm(train_dataloader, desc="Iteration")):
        batch = tuple(t.to(device) for t in batch)
        input_ids, input_mask, segment_ids, label_ids = batch
        loss = model(input_ids, segment_ids, input_mask, label_ids)
        if n_gpu > 1:
            loss = loss.mean() # mean() to average on multi-gpu.
        if gradient_accumulation_steps > 1:
            loss = loss / gradient_accumulation_steps

        if fp16:
            optimizer.backward(loss)
        else:
            loss.backward()

        t.set_description('Training (epoch: {}, step: {}, loss: {:.4f})'.format(epoch, step, loss.item()))
        t.refresh() # to show immediately the update
        
        tr_loss += loss.item()
        nb_tr_examples += input_ids.size(0)
        nb_tr_steps += 1
        if (step + 1) % gradient_accumulation_steps == 0:
            # modify learning rate with special warm up BERT uses
            lr_this_step = learning_rate * warmup_linear(global_step/t_total, warmup_proportion)
            for param_group in optimizer.param_groups:
                param_group['lr'] = lr_this_step
            optimizer.step()
            optimizer.zero_grad()
            global_step += 1

Training (epoch: 0, step: 194, loss: 0.2081):   0%|          | 0/2 [02:13<?, ?it/s]


Training (epoch: 0, step: 388, loss: 0.2283):   0%|          | 0/2 [04:26<?, ?it/s]


Training (epoch: 1, step: 116, loss: 0.0404):  50%|█████     | 1/2 [06:29<06:29, 389.69s/it]


Training (epoch: 1, step: 292, loss: 0.0477):  50%|█████     | 1/2 [08:30<08:30, 510.68s/it]


Training (epoch: 1, step: 451, loss: 0.1710):  50%|█████     | 1/2 [10:20<10:20, 620.03s/it]Training beyond specified 't_total'. Learning rate multiplier set to 0.0. Please set 't_total' of WarmupLinearSchedule correctly.

Training (epoch: 1, step: 451, loss: 0.1710): 100%|██████████| 2/2 [10:20<00:00, 310.07s/it]


the name of our model will have following format
* `model_dir_name`: ``bert-{`BATCH_SIZE`}-{`randidx`}.bin
* where `randidx` is a random index number to identify models with same `BATCH_SIZE` and `MAX_LEN` values

In [37]:
randidx = '{}'.format(np.random.randint(0, 10000)).zfill(4)
model_name_with_fullpath = '{}/bert-{}-{}-{}.bin'.format(MODEL_DIR, BATCH_SIZE, MAX_LEN, randidx)
model_name_with_fullpath

'/data/jigsaw/model/bert-32-80-2015.bin'

In [38]:
# Save a trained model
model_to_save = model.module if hasattr(model, 'module') else model  # Only save the model it-self
#output_model_file = os.path.join('/data/jigsaw/model/{}'.format(model_dir_name), "pytorch_model.bin")
torch.save(model_to_save.state_dict(), model_name_with_fullpath)

<b> Evaluate model <b>

In [39]:
import numpy as np
import pandas as pd
from sklearn.metrics import recall_score
from sklearn.metrics import accuracy_score
from sklearn.metrics import precision_score

In [40]:
SUBGROUP_AUC = 'subgroup_auc'
BPSN_AUC = 'bpsn_auc'  # stands for background positive, subgroup negative
BNSP_AUC = 'bnsp_auc'  # stands for background negative, subgroup positive

In `igsaw Unintended Bias in Toxicity Classification` competition, the objective is to reduce biased decision from some words (such as `black`, `white` or `resbian` and etc) <br>
To reduce bias, the competition host defined a customized metric to evaluate model.<br>
The metric introduced from the following link and defined with following functions.
* https://www.kaggle.com/dborkan/benchmark-kernel

In [41]:
def compute_auc(y_true, y_pred):
    try:
        return sklearn.metrics.roc_auc_score(y_true, y_pred)
    except ValueError:
        return np.nan

def compute_subgroup_auc(df, subgroup, label, model_name):
    subgroup_examples = df[df[subgroup]]
    return compute_auc(subgroup_examples[label], subgroup_examples[model_name])

def compute_bpsn_auc(df, subgroup, label, model_name):
    """Computes the AUC of the within-subgroup negative examples and the background positive examples."""
    subgroup_negative_examples = df[df[subgroup] & ~df[label]]
    non_subgroup_positive_examples = df[~df[subgroup] & df[label]]
    examples = subgroup_negative_examples.append(non_subgroup_positive_examples)
    return compute_auc(examples[label], examples[model_name])

def compute_bnsp_auc(df, subgroup, label, model_name):
    """Computes the AUC of the within-subgroup positive examples and the background negative examples."""
    subgroup_positive_examples = df[df[subgroup] & df[label]]
    non_subgroup_negative_examples = df[~df[subgroup] & ~df[label]]
    examples = subgroup_positive_examples.append(non_subgroup_negative_examples)
    return compute_auc(examples[label], examples[model_name])
    
def compute_bias_metrics_for_model(dataset,
                                   subgroups,
                                   model,
                                   label_col,
                                   include_asegs=False):
    """Computes per-subgroup metrics for all subgroups and one model."""
    records = []
    for subgroup in subgroups:
        record = {
            'subgroup': subgroup,
            'subgroup_size': len(dataset[dataset[subgroup]])
        }
        record[SUBGROUP_AUC] = compute_subgroup_auc(dataset, subgroup, label_col, model)
        record[BPSN_AUC] = compute_bpsn_auc(dataset, subgroup, label_col, model)
        record[BNSP_AUC] = compute_bnsp_auc(dataset, subgroup, label_col, model)
        records.append(record)
    return pd.DataFrame(records).sort_values('subgroup_auc', ascending=True)
    
# Convert taget and identity columns to booleans
def convert_to_bool(df, col_name):
    df[col_name] = np.where(df[col_name] >= 0.5, True, False)
    
def convert_dataframe_to_bool(df):
    bool_df = df.copy()
    for col in ['target'] + identity_columns:
        convert_to_bool(bool_df, col)
    return bool_df

def calculate_overall_auc(df, model_name):
    true_labels = df['target']
    predicted_labels = df[model_name]
    return sklearn.metrics.roc_auc_score(true_labels, predicted_labels)

def power_mean(series, p):
    total = sum(np.power(series, p))
    return np.power(total / len(series), 1 / p)

def get_final_metric(bias_df, overall_auc, POWER=-5, OVERALL_MODEL_WEIGHT=0.25):
    bias_score = np.average([
        power_mean(bias_df[SUBGROUP_AUC], POWER),
        power_mean(bias_df[BPSN_AUC], POWER),
        power_mean(bias_df[BNSP_AUC], POWER)
    ])
    return (OVERALL_MODEL_WEIGHT * overall_auc) + ((1 - OVERALL_MODEL_WEIGHT) * bias_score)
    
def calculate_bias_metrics(df):
    df = convert_dataframe_to_bool(df)
    #_tmp = df.vectorized.values
    #test_padded_vector = pad_sequences(_tmp, padding='post', maxlen=MAX_LEN)
    #df.loc[:,'prediction'] = model.predict(test_padded_vector, batch_size=2048)[0].flatten()

    bias_metrics_valid = compute_bias_metrics_for_model(df, identity_columns, 'prediction', 'target')
    aux = get_final_metric(bias_metrics_valid, calculate_overall_auc(df, 'prediction'))
    return aux, bias_metrics_valid

In [42]:
def calculate_metrics(pred, labels):
    pred = np.argmax(pred, axis=1)
    acc = accuracy_score(labels, pred)
    pre = precision_score(labels, pred)
    rec = recall_score(labels, pred)
    return [acc, pre, rec], pred

load pretrained model (`model_name_with_fullpath`)

In [43]:
# Load a trained model that you have fine-tuned
model_state_dict = torch.load(model_name_with_fullpath)
loaded_model = BertForSequenceClassification.from_pretrained(bert_model_name, state_dict=model_state_dict, num_labels=num_labels)
loaded_model.to(device)

BertForSequenceClassification(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): BertLayerNorm()
      (dropout): Dropout(p=0.1)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0): BertLayer(
          (attention): BertAttention(
            (self): BertSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): BertLayerNorm()
              (dropout): Dropout(p=0.1)
            )
          )
          (intermediat

build dataloader for `valid_df`

In [47]:
valid_examples = processor.get_examples_from_df(valid_df)
valid_features = convert_examples_to_features(valid_examples, label_list, MAX_LEN, bert_tok)

/usr/local/lib/python3.5/dist-packages/pandas/core/indexing.py:543: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: http://pandas.pydata.org/pandas-docs/stable/indexing.html#indexing-view-versus-copy
  self.obj[item] = s


*** Example ***
guid: dev-1
tokens: [CLS] facts are not a priority at fox news , hello . [SEP]
input_ids: 101 8866 2024 2025 1037 9470 2012 4419 2739 1010 7592 1012 102 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0
input_mask: 1 1 1 1 1 1 1 1 1 1 1 1 1 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0
segment_ids: 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0
label: 0 (id = 0)
--------------------------------
*** Example ***
guid: dev-2
tokens: [CLS] very interesting column . speaking of " pastor " mans ##bridge my choice for news anchor is lu ##ba go ##y . lu ##ba may be a go ##y , but she can pass for jewish . and that ' s what this country needs - some of that ol ' time ju ##dae ##o - christian ethics 

In [51]:
all_input_ids = torch.tensor([f.input_ids for f in valid_features], dtype=torch.long)
all_input_mask = torch.tensor([f.input_mask for f in valid_features], dtype=torch.long)
all_segment_ids = torch.tensor([f.segment_ids for f in valid_features], dtype=torch.long)
all_label_ids = torch.tensor([f.label_id for f in valid_features], dtype=torch.long)
valid_data = TensorDataset(all_input_ids, all_input_mask, all_segment_ids, all_label_ids)

In [54]:
# Run prediction for full data
valid_sampler = SequentialSampler(valid_data)
valid_dataloader = DataLoader(valid_data, sampler=valid_sampler, batch_size=BATCH_SIZE)

actually running evaluation with `loaded_model`

In [55]:
# fit to our physical environment
loaded_model.eval()
loaded_model.to(device)

BertForSequenceClassification(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): BertLayerNorm()
      (dropout): Dropout(p=0.1)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0): BertLayer(
          (attention): BertAttention(
            (self): BertSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): BertLayerNorm()
              (dropout): Dropout(p=0.1)
            )
          )
          (intermediat

In [56]:
count = 0
metrics = []
predictions = []
for input_ids, input_mask, segment_ids, label_ids in tqdm(valid_dataloader, desc="Evaluating"):
    input_ids = input_ids.to(device)
    input_mask = input_mask.to(device)
    segment_ids = segment_ids.to(device)
    label_ids = label_ids.to(device)

    with torch.no_grad():
        tmp_eval_loss = loaded_model(input_ids, segment_ids, input_mask, label_ids)
        logits = loaded_model(input_ids, segment_ids, input_mask)

    logits = logits.detach().cpu().numpy()
    label_ids = label_ids.to('cpu').numpy()
    
    scores, pred = calculate_metrics(logits, label_ids)
    metrics.append(scores)
    predictions.extend(pred)
    
    count += input_ids.size(0)

Evaluating:   9%|▉         | 10/113 [00:03<00:39,  2.64it/s]/usr/local/lib/python3.5/dist-packages/sklearn/metrics/classification.py:1135: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples.
  'precision', 'predicted', average, warn_for)
Evaluating:  12%|█▏        | 14/113 [00:05<00:37,  2.65it/s]/usr/local/lib/python3.5/dist-packages/sklearn/metrics/classification.py:1137: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 due to no true samples.
  'recall', 'true', average, warn_for)
Evaluating: 100%|██████████| 113/113 [00:42<00:00,  2.67it/s]


In [57]:
metrics = np.array(metrics)
metrics.shape

(113, 3)

calculate precision/recall/accuracy metrics and customized auc score

In [58]:
# (0.9823249113475178, 0.8453204154002025, 0.8269946808510638)
df = pd.DataFrame({
    'accuracy': metrics[:,0],
    'precision': metrics[:,1],
    'recall': metrics[:,2]
})
print('accuracy: {:.4f}'.format(df.accuracy.mean()))
print('precision: {:.4f}'.format(df.precision.mean()))
print('recall: {:.4f}'.format(df.recall.mean()))

accuracy: 0.9383
precision: 0.5377
recall: 0.5223


In [59]:
valid_df.loc[:, 'prediction'] = predictions
auc, auc_df = calculate_bias_metrics(valid_df)
'AUC Score: {:.4f}'.format(auc)

'AUC Score: 0.6907'

# Personal work
## Kaggle 이미지 분석
- Title of the competition: quick doodle..
- Skill used:
* Language/Library: Python, Keras, Numpy, Pandas
* Preprocessing: extract relavant features for recognizing hand-drawn pictures
* Algorithm: Xception, InceptionResnet, Mobilenet, Ensemble
* Result: 86th out of 1316 competitors (unofficial)

# Experience
## Anomaly detection
- Objective: Detecting users with abnormal behaviour
- Description: Abnormal behaviour was defined by incoming IPs with automative bots.
- Skills used:


## Attack factor recognition Engine
- Objective: Implementing deep learning based algorithm for recognizing "Attack factor" from incoming security events
- Background: Due to high false positive/negative with regex-based approach
- Skills used:
* Language/Library: Python, Tensorflow, Pandas, Numpy
* Data preprocessing: convert to lower case, tokenizing, word embedding
* Desinged customized CNN+RNN based algorithm
* Deployment: Modulized trained tensorflow model on linux environment by using tensorflow C++ API


## WISA paper
- Objective: 
* Researching machine learning techniques to visualize attack sequences per IP address 
* Publishing a machine learning paper
- Background: No machine learning based visualizing techniques for sequence in security domain
- Skills used: 
* Language/Library: Python, C, MPI programming, matplotlib
* Algorithm: Smith Waterman algorithm, Force Directed graph
* Result: Awarded as a best paper
* Related link: ......


## Inference Engine
- Objective: Make a decision of whether incoming security event is real attack or not, so as to make SoC operation automative
- Background: By automating SoC operation, quality of SoC operation can be enhanced because human analysts can handle a story of security events in deeper and wider area
- Description: By taking advantage of result of attack factor recognition engine which I was fully involved in, 
- Skills used:
* Language/Library: Python, Tensorflow, Pandas, Numpy, sklearn
* System design by implementing different types of ML/DL models.
* Deployment: run with tensorflow serving under docker environment

